# 03. CV 체계 + 사전확률 보정 + 결측 복구 피처

팀원의 라벨 생성 규칙 역추적 발견(`sleep_duration` 6/7 & `stress_level` & `physical_activity_level`, 성능 상한 BA ≈ 0.941)을 바탕으로:

1. 팀 전원이 공유할 **고정 CV 폴드** 생성 (`StratifiedKFold(5, shuffle=True, random_state=42)`)
2. **결측 복구 피처** 추가
   - `physical_activity_level` 결측 ← `step_count` 대리변수
   - `sleep_duration` 결측 ← `sleep_quality` 조건부 분포로 조건부 대치
3. LightGBM 베이스라인 학습 + **사전확률 보정** vs `class_weight='balanced'` 둘 중 하나만 비교선택 (이중보정 금지)
4. OOF balanced accuracy로 최종 전략 결정 + test 예측 생성

커널: **Python (teammate)** (lightgbm 4.6.0 설치된 환경)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import lightgbm as lgb

SEED = 42
N_FOLDS = 5

DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "health_condition"
CLASS_ORDER = ["at-risk", "fit", "unhealthy"]  # alphabetical order used by LabelEncoder

print(train.shape, test.shape)

(690088, 15) (295753, 14)


## 1. 고정 CV 폴드 생성

팀 전원이 같은 검증 기준으로 비교할 수 있도록 `fold` 컬럼을 만들어 저장합니다. 이후 모든 노트북/실험은 이 폴드를 그대로 불러와서 씁니다.

In [2]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

train["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(train, train[TARGET])):
    train.loc[val_idx, "fold"] = fold

print(train["fold"].value_counts().sort_index())

# 폴드 배정만 별도로 저장 (재현 가능하게)
train[["id", "fold"]].to_csv(OUT_DIR / "cv_folds.csv", index=False)
print("saved:", OUT_DIR / "cv_folds.csv")

fold
0    138018
1    138018
2    138018
3    138017
4    138017
Name: count, dtype: int64
saved: ../playground-series-s6e7/processed/cv_folds.csv


## 2. 결측 복구 피처

EDA에서 확인한 대리변수 관계를 실제 피처로 만듭니다. **누수 방지를 위해 모든 기준값(경계, 조건부 분포)은 train 전체가 아니라 매 폴드의 train 쪽에서만 계산**해야 하지만, 여기서는 두 대리변수 모두 카테고리 경계값(tertile 경계, 조건부 분포)이 폴드 간 거의 변하지 않는 안정적인 값이라 우선 전체 train 기준으로 계산 후 fold별 재계산과 비교합니다.

In [3]:
def add_recovery_features(df, step_tertiles, sleep_quality_cond):
    df = df.copy()

    # (1) physical_activity_level 결측 -> step_count 기반 대리 카테고리로 대치
    #     step_count 3분위(하/중/상) <-> sedentary/moderate/active 매핑 (EDA에서 확인한 대응 관계)
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)

    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    # step_count까지 없는 극소수 행은 전체 최빈값(sedentary/moderate 중 표본이 더 많은 쪽)으로 대치
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    # (2) sleep_duration 결측 -> sleep_quality별 sleep_duration 조건부 중앙값으로 대치
    df["sleep_duration_recovered"] = df["sleep_duration"]
    missing_sleep = df["sleep_duration"].isna()
    fallback_median = sleep_quality_cond.get("missing", sleep_quality_cond["average"])
    mapped = df.loc[missing_sleep, "sleep_quality"].map(sleep_quality_cond).fillna(fallback_median)
    df.loc[missing_sleep, "sleep_duration_recovered"] = mapped

    # 결측 플래그(핵심 3피처만 유지 - EDA에서 나머지는 신호 미미하다고 확인됨)
    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)

    return df


# 대리변수 기준값은 train 전체로 계산 (fold 무관하게 안정적인 값)
step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()

print("step_count tertile 경계:", step_tertiles)
print("sleep_quality별 sleep_duration 중앙값:", sleep_quality_cond)

train = add_recovery_features(train, step_tertiles, sleep_quality_cond)
test = add_recovery_features(test, step_tertiles, sleep_quality_cond)

train[["physical_activity_level", "physical_activity_level_recovered",
       "sleep_duration", "sleep_duration_recovered"]].head(10)

step_count tertile 경계: [ 6561. 11029.]
sleep_quality별 sleep_duration 중앙값: {'average': 6.99, 'good': 7.54, 'poor': 6.45, 'missing': np.float64(6.99)}


,physical_activity_level,physical_activity_level_recovered,sleep_duration,sleep_duration_recovered
0,sedentary,sedentary,5.22,5.22
1,moderate,moderate,5.53,5.53
2,active,active,5.29,5.29
3,active,active,4.70,4.70
4,sedentary,sedentary,7.23,7.23
5,sedentary,sedentary,5.11,5.11
6,moderate,moderate,8.21,8.21
7,sedentary,sedentary,7.47,7.47
8,active,active,5.94,5.94
9,active,active,6.97,6.97


## 3. 나머지 전처리 (01_preprocessing과 동일한 규칙, `_recovered` 컬럼 추가)

In [4]:
NUMERIC_COLS = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender"]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

# 수치형: train median
numeric_medians = train[NUMERIC_COLS].median()
for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

# 범주형: 'missing' 카테고리
categorical_cols = list(ORDINAL_COLS.keys()) + NOMINAL_COLS
for df in (train, test):
    for col in categorical_cols:
        df[col] = df[col].fillna("missing")

# 순서형 인코딩
for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    encoder = OrdinalEncoder(categories=[categories])
    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

# 명목형 원-핫
train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)

train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

FEATURE_COLS = NUMERIC_COLS + list(ORDINAL_COLS.keys()) + FLAG_COLS + list(train_ohe.columns)
print(len(FEATURE_COLS), "features:", FEATURE_COLS)

target_encoder = LabelEncoder()
train["target_enc"] = target_encoder.fit_transform(train[TARGET])
print(dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_))))

22 features: ['sleep_duration_recovered', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'stress_level', 'sleep_quality', 'physical_activity_level_recovered', 'smoking_alcohol', 'stress_level_isnull', 'sleep_duration_isnull', 'physical_activity_level_isnull', 'diet_type_balanced', 'diet_type_missing', 'diet_type_non-veg', 'diet_type_veg', 'gender_female', 'gender_male', 'gender_missing', 'gender_other']
{'at-risk': np.int64(0), 'fit': np.int64(1), 'unhealthy': np.int64(2)}


## 4. 사전확률 보정 함수 정의

In [5]:
def prior_corrected_predict(proba, class_order, priors):
    """proba: (n, n_class) 예측 확률. class_order와 priors는 같은 순서.
    balanced accuracy 최적화를 위해 P(c|x)/P(c)가 최대인 클래스를 선택."""
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


train_priors = train[TARGET].value_counts(normalize=True).to_dict()
print("train 사전확률:", train_priors)

train 사전확률: {'at-risk': 0.8586745458550213, 'unhealthy': 0.08364730295266691, 'fit': 0.057678151192311705}


## 5. CV 학습 — 4가지 방식 비교

1. **A. plain argmax**: 일반 학습 + `argmax P(c|x)` (기준선, accuracy는 높지만 BA는 낮을 것으로 예상)
2. **B. 사전확률 보정**: 일반 학습 + `argmax P(c|x)/P(c)`
3. **C. class_weight='balanced'**: 학습 자체에서 보정 + `argmax P(c|x)`
4. **D. 이중보정(주의용 대조군)**: class_weight='balanced' 학습 + 사전확률 보정까지 적용 → 점수가 오히려 떨어지는지 확인용

In [6]:
oof_proba_plain = np.zeros((len(train), 3))
oof_proba_balanced = np.zeros((len(train), 3))

lgb_class_order = list(target_encoder.classes_)  # LightGBM이 출력하는 클래스 순서 (LabelEncoder 알파벳순)

for fold in range(N_FOLDS):
    tr_idx = train["fold"] != fold
    va_idx = train["fold"] == fold

    X_tr, y_tr = train.loc[tr_idx, FEATURE_COLS], train.loc[tr_idx, "target_enc"]
    X_va, y_va = train.loc[va_idx, FEATURE_COLS], train.loc[va_idx, "target_enc"]

    params = dict(
        objective="multiclass", num_class=3, n_estimators=500,
        learning_rate=0.05, num_leaves=63, subsample=0.8,
        colsample_bytree=0.8, random_state=SEED, verbosity=-1,
    )

    model_plain = lgb.LGBMClassifier(**params)
    model_plain.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    oof_proba_plain[va_idx.values] = model_plain.predict_proba(X_va)

    model_balanced = lgb.LGBMClassifier(**params, class_weight="balanced")
    model_balanced.fit(
        X_tr, y_tr, eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    oof_proba_balanced[va_idx.values] = model_balanced.predict_proba(X_va)

    print(f"fold {fold} done")

print("CV 완료")

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done
CV 완료


In [7]:
y_true = train[TARGET].values

pred_A = np.array(lgb_class_order)[oof_proba_plain.argmax(axis=1)]
pred_B = prior_corrected_predict(oof_proba_plain, lgb_class_order, train_priors)
pred_C = np.array(lgb_class_order)[oof_proba_balanced.argmax(axis=1)]
pred_D = prior_corrected_predict(oof_proba_balanced, lgb_class_order, train_priors)

results = []
for name, pred in [("A. plain argmax", pred_A),
                    ("B. 사전확률 보정", pred_B),
                    ("C. class_weight=balanced", pred_C),
                    ("D. 이중보정(대조군)", pred_D)]:
    acc = accuracy_score(y_true, pred)
    ba = balanced_accuracy_score(y_true, pred)
    results.append({"방식": name, "accuracy": round(acc, 5), "balanced_accuracy": round(ba, 5)})

results_df = pd.DataFrame(results)
results_df

,방식,accuracy,balanced_accuracy
0,A. plain argmax,0.96706,0.87602
1,B. 사전확률 보정,0.93922,0.94980
2,C. class_weight=balanced,0.94212,0.94919
3,D. 이중보정(대조군),0.85981,0.92517


## 6. 최종 전략 선택 및 test 예측 생성

위 표에서 balanced_accuracy가 가장 높은 방식을 최종 전략으로 채택합니다 (B와 C 중 더 나은 쪽 — D는 이중보정 부작용 확인용 대조군이라 채택하지 않음).

In [8]:
best_row = results_df.iloc[[1, 2]].sort_values("balanced_accuracy", ascending=False).iloc[0]
print("최종 채택 전략:", best_row["방식"])

# 전체 train으로 재학습 (fold 없이) 후 test 예측
use_class_weight = "class_weight" in best_row["방식"]

final_params = dict(
    objective="multiclass", num_class=3, n_estimators=500,
    learning_rate=0.05, num_leaves=63, subsample=0.8,
    colsample_bytree=0.8, random_state=SEED, verbosity=-1,
)
if use_class_weight:
    final_params["class_weight"] = "balanced"

final_model = lgb.LGBMClassifier(**final_params)
final_model.fit(train[FEATURE_COLS], train["target_enc"])

test_proba = final_model.predict_proba(test[FEATURE_COLS])

if use_class_weight:
    test_pred = np.array(lgb_class_order)[test_proba.argmax(axis=1)]
else:
    test_pred = prior_corrected_predict(test_proba, lgb_class_order, train_priors)

submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
submission.to_csv(OUT_DIR / "submission_v1.csv", index=False)
print("saved:", OUT_DIR / "submission_v1.csv")
submission[TARGET].value_counts(normalize=True)

최종 채택 전략: B. 사전확률 보정


saved: ../playground-series-s6e7/processed/submission_v1.csv


health_condition
at-risk      0.810852
unhealthy    0.115306
fit          0.073842
Name: proportion, dtype: float64

## 7. Feature Importance

In [9]:
importance = pd.Series(final_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
importance

bmi                                  12666
sleep_duration_recovered             11062
water_intake                         11010
heart_rate                           10752
exercise_duration                    10671
step_count                           10286
calorie_expenditure                  10042
stress_level                          3265
smoking_alcohol                       2713
sleep_quality                         2053
physical_activity_level_recovered     1444
sleep_duration_isnull                 1198
physical_activity_level_isnull         895
diet_type_non-veg                      781
gender_male                            764
diet_type_balanced                     759
diet_type_veg                          755
gender_other                           665
gender_female                          597
gender_missing                         308
stress_level_isnull                    194
diet_type_missing                      120
dtype: int32